# 🚁 Notebook 05 — Integral Control

### Erasing the stubborn steady-state error

At the end of Notebook 04 we hit a wall: a pure **P** controller always settles a little *below*
the target, because it needs a permanent error to make the thrust that fights gravity. Turning `Kp`
up shrinks the gap but never closes it (and causes oscillation).

This notebook closes it — exactly — with the **I** in PID: **Integral** control. The idea:

> **Keep a running total of the error over time. If we've been short for a while, that total grows,
> and we add extra thrust until the error becomes zero.**

It's a patient bookkeeper that refuses to let a small error persist. We'll also meet its famous
failure mode — **windup** — and the standard cure, **anti-windup**.

---

## 🎯 Learning objectives
- Understand *why* accumulating error removes steady-state error.
- Build a **PI controller** and watch the error driven to **exactly zero**.
- *See* the integral term slowly ramp up to cancel gravity.
- Diagnose **integral windup** and fix it with **anti-windup (clamping)**.

## 🧩 Prerequisites
Notebooks 01–04 (especially P control and the steady-state error `m·g/Kp`).

## ⏱️ Estimated time
About **50–65 minutes**.

## 🧠 Concepts you know so far
Euler sim · forces/gravity/thrust · feedback · error · **P control (Kp), overshoot, steady-state error**

## 🔜 Concepts you'll learn here
Integral accumulation · integral gain (Ki) · PI controller · windup · anti-windup / clamping


### 🔁 Quick recap
A P controller does `thrust = Kp·error`. To hover it needs `thrust = m·g`, which forces
`error = m·g/Kp ≠ 0` — a permanent gap. We now add a term that *builds up over time* to supply that
`m·g` on its own, letting the error fall to zero. Run setup, then the reusable simulator cell.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


### 🧰 Our reusable simulator (tidy version of Notebook 04's loop)

To stop rewriting the drone loop, here's a small `simulate()` engine we'll reuse for the rest of
the course. It runs the drone with **any** controller you give it (a controller is just something
that takes `(target, measured, dt)` and returns a thrust). It also has switches for noise, wind,
delay, and thrust limits that we'll use in later notebooks. Read it once — it's exactly the physics
you already know.


In [ ]:
from collections import deque

def simulate(controller, target=10.0, mass=1.0, g=9.8, start_height=0.0,
             dt=0.02, total_time=12.0, max_thrust=30.0, min_thrust=0.0,
             noise_std=0.0, wind=0.0, delay_steps=0, seed=0):
    """Run the 1D drone with ANY controller and return recorded signals as arrays.

    `controller` is a callable: controller(target, measured_altitude, dt) -> desired_thrust.
    It may hold internal state (integral, previous error, etc.) and may expose a dict
    `controller.last_terms = {'p':..,'i':..,'d':..}` which we record for the dashboards.

    Physical extras (all optional, default off):
      noise_std  : Gaussian sensor noise standard deviation (metres)
      wind       : constant extra force in newtons (+ up, - down)
      delay_steps: how many steps the sensor reading lags behind reality
      max/min_thrust : actuator saturation limits (newtons)
    """
    rng = np.random.default_rng(seed)
    x, v, t = start_height, 0.0, 0.0
    buf = deque([start_height] * (delay_steps + 1), maxlen=delay_steps + 1)  # sensor delay line

    keys = ("t", "x", "v", "a", "measured", "thrust", "error", "p", "i", "d")
    hist = {k: [] for k in keys}

    for _ in range(int(total_time / dt)):
        buf.append(x)                                    # newest true altitude enters the line
        delayed = buf[0]                                 # controller sees the OLD reading
        measured = delayed + (rng.normal(0, noise_std) if noise_std > 0 else 0.0)
        error = target - measured

        thrust = controller(target, measured, dt)        # <-- the controller decides
        thrust = min(max(thrust, min_thrust), max_thrust)  # actuator can't exceed limits

        net_force = thrust - mass * g + wind             # sum of forces (up +, down -)
        a = net_force / mass                             # Newton's second law

        terms = getattr(controller, "last_terms", {"p": 0.0, "i": 0.0, "d": 0.0})
        for k, val in zip(keys, (t, x, v, a, measured, thrust, error,
                                 terms.get("p", 0.0), terms.get("i", 0.0), terms.get("d", 0.0))):
            hist[k].append(val)

        v = v + a * dt                                   # Euler integrate
        x = x + v * dt
        if x < 0:                                        # ground
            x, v = 0.0, 0.0
        t = t + dt

    return {k: np.array(val) for k, val in hist.items()}


## 1. The integral idea (intuition)

Picture a manager watching a worker who is *consistently* finishing 2 tasks short every day. A pure
P manager reacts only to *today's* shortfall. An **integral** manager keeps a **running total** of
all the shortfalls, and the longer the deficit persists, the more pressure they apply — until the
backlog is cleared and the total stops growing.

For the drone: we add up the error at every step (error × dt, accumulated). While the drone sits
below target, this **accumulated error keeps growing**, adding more and more thrust — until the
drone reaches the target and the error becomes zero, at which point the accumulator *stops growing*
and holds exactly the thrust needed to hover.

$$ \text{thrust} = \underbrace{K_p \cdot \text{error}}_{\text{P: now}} \;+\; \underbrace{K_i \cdot \!\!\sum (\text{error}\times dt)}_{\text{I: history}} $$

`Ki` is the **integral gain** — how strongly the accumulated history pushes. Let's build it.


## 2. Build the PI controller and compare to P-only

Our controller is now a small object that remembers its running total between steps. Watch the PI
drone reach the target *exactly*, while the P-only drone stalls below it.


In [ ]:
class PController:
    """Pure proportional controller (for comparison)."""
    def __init__(self, Kp):
        self.Kp = Kp
    def __call__(self, target, measured, dt):
        return self.Kp * (target - measured)

class PIController:
    """Proportional + Integral. Remembers the running total of error."""
    def __init__(self, Kp, Ki):
        self.Kp, self.Ki = Kp, Ki
        self.integral = 0.0                     # the running total starts at zero
    def __call__(self, target, measured, dt):
        error = target - measured
        self.integral += error * dt             # accumulate error over time  <-- the I idea
        return self.Kp * error + self.Ki * self.integral

# Run both with the same Kp:
res_P  = simulate(PController(Kp=3.0),              total_time=20.0)
res_PI = simulate(PIController(Kp=3.0, Ki=1.5),     total_time=20.0)

plt.figure(figsize=(9, 5))
plt.plot(res_P["t"],  res_P["x"],  color="tab:orange", lw=2, label="P only  (stalls below)")
plt.plot(res_PI["t"], res_PI["x"], color="tab:blue",   lw=2, label="PI  (reaches target exactly)")
plt.axhline(10, color="seagreen", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Integral action erases the steady-state error")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

print(f"P-only  final altitude: {res_P['x'][-500:].mean():.3f} m  (short of 10)")
print(f"PI      final altitude: {res_PI['x'][-500:].mean():.3f} m  (right on 10!)")


## 3. *Watch* the integral do its job

The magic is easier to believe when you see it. Below, the top panel is altitude; the bottom shows
the **integral term's contribution to thrust** growing over time. Notice: the integral climbs while
the drone is below target, and **flattens out at exactly the value needed to hover (≈ m·g = 9.8 N)**
once the error hits zero. That flattening *is* the steady-state error being gone.


In [ ]:
ctrl = PIController(Kp=3.0, Ki=1.5)
res = simulate(ctrl, total_time=20.0)
# recompute the integral term over time for plotting
integral_term = []
c2 = PIController(Kp=3.0, Ki=1.5)
for m in res["measured"]:
    c2(10.0, m, 0.02)
    integral_term.append(c2.Ki * c2.integral)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
a1.plot(res["t"], res["x"], color="tab:blue", lw=2); a1.axhline(10, color="seagreen", ls="--")
a1.set_ylabel("altitude (m)"); a1.set_title("PI: altitude reaches target"); a1.grid(True, alpha=0.3)
a2.plot(res["t"], integral_term, color="tab:purple", lw=2)
a2.axhline(9.8, color="gray", ls=":", label="thrust needed to hover (m*g)")
a2.set_ylabel("integral term (N)"); a2.set_xlabel("time (s)")
a2.set_title("The integral ramps up, then holds exactly m*g to cancel gravity")
a2.legend(); a2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 4. 🎬 PI drone reaching the exact target


In [ ]:
res = simulate(PIController(Kp=3.0, Ki=1.5), total_time=16.0)
t, x, thr = res["t"], res["x"], res["thrust"]
fids = np.linspace(0, len(t)-1, 120).astype(int)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1); ax.set_ylim(0, 13); ax.set_xticks([])
ax.set_ylabel("altitude (m)"); ax.set_title("PI control")
ax.axhline(0, color="saddlebrown", lw=3); ax.axhline(10, color="seagreen", ls="--", lw=2)
drone, = ax.plot([], [], "o", color="tab:blue", markersize=26)
label = ax.text(-0.9, 12, "", fontsize=11)
def init(): drone.set_data([], []); label.set_text(""); return drone, label
def update(f):
    i = fids[f]; drone.set_data([0], [x[i]])
    label.set_text(f"t={t[i]:.1f}s\nx={x[i]:.2f} m\nthrust={thr[i]:.1f} N"); return drone, label
ani = animation.FuncAnimation(fig, update, frames=len(fids), init_func=init, blit=True, interval=45)
plt.close(fig); HTML(ani.to_jshtml())


## 5. The dark side: **integral windup**

Integral action has a dangerous failure mode. Real motors have a **maximum thrust** (they saturate).
Suppose the drone must climb a long way and the motors are already maxed out. The drone still isn't
at target, so the error stays positive, so **the integral keeps accumulating** — even though extra
thrust can't do anything (the motor is already flat out).

By the time the drone finally reaches the target, the integral has grown *enormous*. It now
commands huge thrust and the drone **blasts far past the target** before the bloated integral
unwinds. This overshoot-from-a-saturated-integral is called **windup**. Let's cause it on purpose:
low `max_thrust`, big climb.


In [ ]:
# Force saturation: motors max out at 13 N while climbing to a high target from the ground.
res_windup = simulate(PIController(Kp=2.0, Ki=1.2), target=10.0,
                       max_thrust=13.0, total_time=30.0)

plt.figure(figsize=(9, 5))
plt.plot(res_windup["t"], res_windup["x"], color="tab:red", lw=2, label="PI (no anti-windup)")
plt.axhline(10, color="seagreen", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("WINDUP: saturated motors let the integral balloon -> massive overshoot")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("The drone rockets way past 10 m and takes ages to recover, because the integral")
print("kept growing while the motors were maxed out and useless.")


## 6. The fix: **anti-windup** (clamp the integral)

The cure is simple: **don't let the integral grow without bound.** We *clamp* it to a sensible
range, so it can't accumulate a monster value while the motors are saturated. This one change tames
the overshoot completely.


In [ ]:
class PIController_AntiWindup:
    def __init__(self, Kp, Ki, i_limit=12.0):
        self.Kp, self.Ki = Kp, Ki
        self.integral = 0.0
        self.i_limit = i_limit               # max newtons the integral term may contribute
    def __call__(self, target, measured, dt):
        error = target - measured
        self.integral += error * dt
        # ANTI-WINDUP: clamp so Ki*integral stays within +/- i_limit
        cap = self.i_limit / self.Ki
        self.integral = max(-cap, min(cap, self.integral))
        return self.Kp * error + self.Ki * self.integral

res_fixed = simulate(PIController_AntiWindup(Kp=2.0, Ki=1.2, i_limit=12.0),
                     target=10.0, max_thrust=13.0, total_time=30.0)

plt.figure(figsize=(9, 5))
plt.plot(res_windup["t"], res_windup["x"], color="tab:red",  lw=2, alpha=0.6, label="no anti-windup")
plt.plot(res_fixed["t"],  res_fixed["x"],  color="tab:green", lw=2, label="with anti-windup")
plt.axhline(10, color="seagreen", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Anti-windup (clamping the integral) removes the huge overshoot")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("Same saturated motors, same gains -- but clamping the integral keeps it civilized.")


## 7. 🎛️ Play: Ki, and anti-windup on/off

Turn `Ki` up and down (too much causes slow, rolling oscillations as the integral over-corrects).
Lower `max_thrust` to force saturation, then toggle anti-windup to see it rescue the response.


In [ ]:
def explore_PI(Kp=2.0, Ki=1.2, max_thrust=13.0, anti_windup=True):
    if anti_windup:
        ctrl = PIController_AntiWindup(Kp, Ki, i_limit=12.0)
    else:
        ctrl = PIController(Kp, Ki)
    res = simulate(ctrl, target=10.0, max_thrust=max_thrust, total_time=30.0)
    plt.figure(figsize=(9, 4.5))
    plt.plot(res["t"], res["x"], color=("tab:green" if anti_windup else "tab:red"), lw=2)
    plt.axhline(10, color="seagreen", ls="--"); plt.ylim(0, 22)
    plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
    plt.title(f"Kp={Kp}, Ki={Ki}, max_thrust={max_thrust} N, anti-windup={anti_windup}")
    plt.grid(True, alpha=0.3); plt.show()

interact(explore_PI,
         Kp=FloatSlider(min=0.5, max=6.0, step=0.5, value=2.0),
         Ki=FloatSlider(min=0.0, max=5.0, step=0.2, value=1.2),
         max_thrust=FloatSlider(min=11.0, max=30.0, step=1.0, value=13.0),
         anti_windup=Checkbox(value=True, description="anti-windup"));


## ✅ Summary
- **Integral control** accumulates error over time: `I contribution = Ki × Σ(error·dt)`.
- It **removes steady-state error**: the integral ramps up until it supplies exactly the thrust
  needed to hover (≈ m·g), at which point the error is zero and it stops growing.
- Too much `Ki` makes the loop sluggish and prone to slow oscillation.
- **Windup**: when actuators saturate, the integral balloons and causes huge overshoot.
- **Anti-windup** (clamping the integral) is the standard, simple fix.


## ⚠️ Common mistakes
- **Ki too high.** It doesn't reach target faster; it destabilizes with rolling oscillations.
- **Forgetting anti-windup.** Any real system with thrust limits *will* wind up eventually.
- **Expecting I to react fast.** Integral is patient and slow by nature — it fixes *persistent*
  error, not sudden shocks (that's D's job, next).

## 🧭 Mental model
> *"P is a spring; I is a patient helper who keeps a ledger of how long you've been off-target and
> pushes harder the longer it lasts — until the ledger, not the error, holds the drone up."*

## 🌍 Real-world applications
Thermostats reaching the exact set temperature, car cruise control holding speed uphill, chemical
plant level control, and every real flight controller's altitude/attitude loops.


## 🧪 Exercises

**L1 — Observe.** In Section 3, what value does the integral term flatten out to, and why that
number?
<details><summary>▸ Solution</summary>
About <b>9.8 N</b> — exactly <code>m·g</code>, the thrust needed to hover. Once the integral alone
supplies that, the P term's contribution goes to zero (error = 0) and the integral stops growing.
</details>

**L2 — Modify.** Add a constant `wind = -3.0` N to a `simulate(...)` call for the PI controller.
Does it *still* reach the target exactly? What did the integral do?
<details><summary>▸ Solution</summary>
Yes, still exactly on target. The integral simply grows to a larger value to supply
<code>m·g + 3</code> N. Rejecting constant disturbances with zero steady-state error is integral
control's superpower.
</details>

**L3 — Predict.** With saturation at `max_thrust=13` and *no* anti-windup, will a *larger* `Ki`
make the windup overshoot worse or better? Predict, then test.
<details><summary>▸ Solution</summary>
**Worse.** Larger Ki accumulates the (bloated) integral faster during saturation, so the overshoot
grows. Anti-windup is what makes larger Ki safe.
</details>


## ❓ Quick quiz
**Q1.** The integral term removes steady-state error by…
<details><summary>▸ Answer</summary>Accumulating error over time until it supplies the exact thrust needed to hover, driving error to zero.</details>

**Q2.** Windup happens when…
<details><summary>▸ Answer</summary>The actuator **saturates** (maxes out) while error persists, so the integral keeps growing uselessly and then causes big overshoot.</details>

**Q3.** Anti-windup works by…
<details><summary>▸ Answer</summary>**Clamping** the integral so it can't grow without bound.</details>

**Q4.** Compared to P, the integral term is…
<details><summary>▸ Answer</summary>**Slower / more patient** — it fixes persistent error, not sudden changes.</details>


## 🔭 Preview of Notebook 06 — *Derivative Control*

P reacts to the error **now**; I reacts to the error's **past**. Neither *anticipates* the future,
so the drone still overshoots and oscillates. The **Derivative (D)** term watches how *fast* the
error is changing and acts like a **shock absorber**, braking before we overshoot. But D has a
notorious weakness — it **amplifies sensor noise** — so we'll also learn to filter it. 🚁
